In [1]:
import site

In [2]:
site.getsitepackages()

['/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12/site-packages']

In [3]:
import sys

In [4]:
print(sys.path)

['/group/pmc021/amunif/epi-thesis/workflow/09_Learning to Rank', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python312.zip', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12/lib-dynload', '', '/home/amunif/.local/lib/python3.12/site-packages', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12/site-packages']


In [5]:
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python311.zip')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/lib-dynload')
sys.path.append('/home/amunif/.local/lib/python3.11/site-packages')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages')

In [6]:
print(sys.path)

['/group/pmc021/amunif/epi-thesis/workflow/09_Learning to Rank', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python312.zip', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12/lib-dynload', '', '/home/amunif/.local/lib/python3.12/site-packages', '/uwahpc/centos8/python/Anaconda3/2024.06/lib/python3.12/site-packages', '/group/pmc021/amunif/env/pytorch/lib/python311.zip', '/group/pmc021/amunif/env/pytorch/lib/python3.11', '/group/pmc021/amunif/env/pytorch/lib/python3.11/lib-dynload', '/home/amunif/.local/lib/python3.11/site-packages', '/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages']


In [7]:
from sklearn.datasets import make_classification
import numpy as np
import xgboost as xgb

In [8]:
# Create a synthetic ranking dataset
seed = 1994
X, y = make_classification(random_state=seed)
rng = np.random.default_rng(seed)
n_query_groups = 5
qid = rng.integers(0, n_query_groups, size=X.shape[0])

In [9]:
X

array([[ 1.37274475,  0.55556022, -0.39472313, ..., -1.52678928,
        -0.95222834, -0.58304118],
       [ 0.29797914,  0.60418421,  0.24888156, ..., -0.68180427,
        -0.81937699, -1.8177727 ],
       [-0.20681544,  0.18770098, -0.31246672, ..., -1.71367001,
        -1.77090831, -0.37479097],
       ...,
       [-0.57437761, -1.66850427,  1.18792669, ..., -0.31096033,
         0.80405855,  0.25831555],
       [ 0.86229379,  0.61644239,  1.7479823 , ..., -0.23655313,
        -1.26747288, -0.20751249],
       [-0.34736961, -0.81172299, -0.21660061, ...,  0.05799518,
        -0.45621093,  0.36914458]])

In [10]:
y

array([0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0])

In [11]:
qid

array([3, 3, 4, 0, 1, 2, 3, 4, 4, 1, 2, 3, 4, 1, 2, 2, 4, 2, 1, 3, 2, 2,
       4, 0, 2, 3, 4, 3, 2, 1, 0, 1, 0, 2, 0, 0, 3, 3, 3, 4, 0, 3, 2, 2,
       4, 2, 2, 2, 0, 3, 4, 4, 3, 1, 0, 0, 3, 3, 0, 3, 1, 2, 4, 1, 3, 4,
       1, 0, 2, 4, 1, 4, 3, 1, 3, 2, 0, 1, 1, 4, 3, 1, 4, 2, 3, 1, 2, 3,
       2, 3, 3, 1, 4, 4, 1, 3, 3, 0, 3, 3])

In [12]:
# Sort the inputs based on query index
sorted_idx = np.argsort(qid)
X = X[sorted_idx, :]
y = y[sorted_idx]
qid = qid[sorted_idx]

In [13]:
# Create and train the XGBRanker model
ranker = xgb.XGBRanker(
    tree_method="hist",
    lambdarank_num_pair_per_sample=8,
    objective="rank:ndcg",
    lambdarank_pair_method="topk"
)
ranker.fit(X, y, qid=qid)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [14]:
# Make predictions
scores = ranker.predict(X)

In [15]:
# Sort the relevance scores from most relevant to least relevant
sorted_idx = np.argsort(scores)[::-1]
scores = scores[sorted_idx]

In [16]:
scores

array([ 2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,
        2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,
        2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,
        2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,  2.079273  ,
        2.079273  ,  1.9736174 ,  1.6827937 ,  1.6827937 ,  1.6827937 ,
        1.588201  ,  1.588201  ,  1.588201  ,  1.5788579 ,  1.5286697 ,
        1.5033935 ,  1.4680612 ,  1.388395  ,  1.3579859 ,  1.2973773 ,
        1.2692909 ,  1.1491511 ,  0.51418805,  0.42684755,  0.11012257,
        0.06206822, -0.04030252, -0.19293904, -0.6285441 , -0.78864837,
       -0.8805593 , -0.8841968 , -0.8878853 , -0.9213908 , -0.99242663,
       -1.0301739 , -1.0541493 , -1.0555627 , -1.1173434 , -1.2796494 ,
       -1.3333683 , -1.3402578 , -1.3402578 , -1.36029   , -1.4058574 ,
       -1.4418876 , -1.4421513 , -1.4956741 , -1.4956741 , -1.5227922 ,
       -1.5437813 , -1.5437813 , -1.5562826 , -1.5562826 , -1.56